<a href="https://colab.research.google.com/github/Rohit4507/gpt2-style-emulation/blob/main/gpt2_style_emulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets sentence-transformers accelerate evaluate matplotlib seaborn tqdm


In [ ]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns


from datasets import load_dataset, Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
!pip install -U datasets huggingface_hub

In [ ]:
dataset = load_dataset("fancyzhx/ag_news", split="train")

print("Total samples:", len(dataset))
print(dataset[0])


In [ ]:
authors = {}

for label in range(4):  # use 3 categories
    texts = [x["text"] for x in dataset if x["label"] == label]
    texts = [t for t in texts if 20 < len(t) <= 200]
    authors[f"author_{label}"] = texts[:300]

for k in authors:
    print(k, len(authors[k]))


In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


In [ ]:
def build_dataset(texts):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=64
    )
    return Dataset.from_dict(encodings)


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_SAVE_DIR = "/content/drive/MyDrive/micro_style_models"
os.makedirs(BASE_SAVE_DIR, exist_ok=True)


def fine_tune_or_load(texts, author_name):

    save_path = os.path.join(BASE_SAVE_DIR, author_name)

    # ✅ If model already exists → LOAD
    if os.path.exists(save_path):
        print(f"Loading saved model for {author_name}")
        model = GPT2LMHeadModel.from_pretrained(save_path)
        model.to(device)
        return model

    # ✅ Otherwise → TRAIN
    print(f"Training new model for {author_name}")

    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.to(device)

    train_dataset = build_dataset(texts)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    training_args = TrainingArguments(
        output_dir="./temp_training",
        per_device_train_batch_size=8,
        num_train_epochs=2,
        save_strategy="no",
        logging_steps=100,
        learning_rate=5e-5,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator
    )

    trainer.train()

    # ✅ SAVE MODEL TO DRIVE
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    print(f"Model saved for {author_name}")

    return model


In [ ]:
DATA_SIZES = [25, 50, 100]
trained_models = {}

for author in authors:
    trained_models[author] = {}

    for size in DATA_SIZES:

        model_name = f"{author}_{size}"

        print(f"Processing {model_name}")

        trained_models[author][size] = fine_tune_or_load(
            authors[author][:size],
            model_name
        )


In [ ]:
def generate_text(model, prompt=""):
    if not prompt:
        # If prompt is empty, start generation with the EOS token (or BOS if defined)
        # GPT2 typically doesn't have a BOS token, so EOS token is often used as a start token for generation
        input_ids = torch.full((1, 1), tokenizer.eos_token_id, dtype=torch.long, device=device)
        attention_mask = torch.ones(input_ids.shape, dtype=torch.long, device=device)
        inputs = {"input_ids": input_ids, "attention_mask": attention_mask}
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import json
from tqdm.auto import tqdm  # safe even if already imported

# Directory for saving generated texts
GEN_SAVE_DIR = "/content/drive/MyDrive/micro_style_generations"
os.makedirs(GEN_SAVE_DIR, exist_ok=True)

generated = {}

for author in authors:
    generated[author] = {}

    for size in DATA_SIZES:

        file_path = os.path.join(
            GEN_SAVE_DIR,
            f"{author}_{size}.json"
        )

        should_generate = True
        # ✅ If file exists → LOAD
        if os.path.exists(file_path):
            try:
                print(f"Loading generated texts for {author}_{size}")
                with open(file_path, "r") as f:
                    generated[author][size] = json.load(f)
                should_generate = False
            except json.JSONDecodeError:
                print(f"Corrupted or empty JSON file found for {author}_{size}. Regenerating.")
                # If file is corrupted, we proceed to generate

        # ✅ Otherwise or if corrupted → GENERATE with progress bar
        if should_generate:
            print(f"Generating texts for {author}_{size}")

            texts = []

            for _ in tqdm(
                range(20),
                desc=f"{author}_{size}",
                leave=False
            ):
                text = generate_text(trained_models[author][size])
                texts.append(text)

            generated[author][size] = texts

            with open(file_path, "w") as f:
                json.dump(texts, f)

            print(f"Saved generations for {author}_{size}")

        print(f"Generated texts for {author}_{size}:")
        for i, text in enumerate(generated[author][size]):
            print(f"  {i+1}. {text}")
        print("\n")

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
def style_similarity(real, gen):
    real_emb = embedder.encode(real)
    gen_emb = embedder.encode(gen)
    sims = cosine_similarity(gen_emb, real_emb)
    return sims.mean()


In [ ]:
scores = {}

for author in authors:
    scores[author] = []
    for size in DATA_SIZES:
        score = style_similarity(
            authors[author][150:250],
            generated[author][size]
        )
        scores[author].append(score)

plt.figure(figsize=(8,5))

for author in authors:
    sns.lineplot(x=DATA_SIZES, y=scores[author], marker="o", label=author)

plt.xlabel("Training Samples")
plt.ylabel("Style Similarity")
plt.title("Style Capacity Curve")
plt.grid(True)
plt.show()


In [ ]:
author_list = list(authors.keys())
n = len(author_list)

matrix = np.zeros((n, n))

for i, a in enumerate(author_list):
    gen = generated[a][100]
    for j, b in enumerate(author_list):
        matrix[i][j] = style_similarity(
            authors[b][150:250],
            gen
        )

plt.figure(figsize=(7,6))
sns.heatmap(matrix, annot=True,
            xticklabels=author_list,
            yticklabels=author_list)

plt.xlabel("Compared Author")
plt.ylabel("Generated From")
plt.title("Style Confusion Matrix")
plt.show()
